# ClearFrame — Free Colab T4 GPU Upscaler (ComfyUI + RealESRGAN)

Run ClearFrame's optional **AI Upscaler (GPU)** mode 100% free using Google Colab's T4 GPU:

1. In Colab, open **Runtime → Change runtime type → T4 GPU → Save**.
2. Run **Cell 1** below to install ComfyUI, `ComfyUI-VideoHelperSuite`, `RealESRGAN_x2plus`, and `cloudflared` (~2–3 minutes, first run only).
3. Run **Cell 2** below to start ComfyUI + the free Cloudflare quick tunnel.
4. Copy the printed `https://....trycloudflare.com` URL and paste it into ClearFrame's **AI Upscaler (GPU)** box → click **Connect**.

> Keep Cell 2 running while upscaling. When finished, simply stop Cell 2 or close Colab — your URL is never saved beyond your browser session.

In [ ]:
#@title Cell 1 — Install ComfyUI, VideoHelperSuite, RealESRGAN_x2plus & cloudflared
import os, subprocess, sys

gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise RuntimeError("No NVIDIA GPU detected! In Colab, go to Runtime -> Change runtime type -> T4 GPU, then run Cell 1 again.")
print("Detected GPU:", gpu_check.stdout.strip())

if not os.path.exists("/content/ComfyUI"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", "/content/ComfyUI"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/ComfyUI/requirements.txt"], check=True)

vhs_dir = "/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite"
if not os.path.exists(vhs_dir):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git", vhs_dir], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{vhs_dir}/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"], check=True)

model_dir = "/content/ComfyUI/models/upscale_models"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "RealESRGAN_x2plus.pth")
if not os.path.exists(model_path) or os.path.getsize(model_path) < 10_000_000:
    print("Downloading RealESRGAN_x2plus.pth (67 MB)...")
    subprocess.run(["curl", "-fL", "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth", "-o", model_path], check=True)

cloudflared_bin = "/usr/local/bin/cloudflared"
if not os.path.exists(cloudflared_bin):
    print("Downloading cloudflared...")
    subprocess.run(["curl", "-fL", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", cloudflared_bin], check=True)
    os.chmod(cloudflared_bin, 0o755)

print("\nSetup complete! Now run Cell 2 below to start ComfyUI and get your tunnel URL.")

In [ ]:
#@title Cell 2 — Start ComfyUI + Free Cloudflared Tunnel & Print URL
import os, re, subprocess, sys, time, urllib.request

# Clean any previous session input/output files to keep disk tidy
for folder in ["/content/ComfyUI/input", "/content/ComfyUI/output"]:
    os.makedirs(folder, exist_ok=True)
    for f in os.listdir(folder):
        fp = os.path.join(folder, f)
        if os.path.isfile(fp):
            try:
                os.remove(fp)
            except OSError:
                pass

comfy_log = open("/tmp/comfyui.log", "w")
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--enable-cors-header", "*"],
    cwd="/content/ComfyUI",
    stdout=comfy_log,
    stderr=subprocess.STDOUT,
)

print("Starting ComfyUI on port 8188...")
ready = False
for _ in range(90):
    if comfy_proc.poll() is not None:
        comfy_log.close()
        with open("/tmp/comfyui.log") as lf:
            print(lf.read()[-2000:])
        raise RuntimeError("ComfyUI exited unexpectedly. Check log output above.")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(2)

if not ready:
    comfy_proc.terminate()
    raise RuntimeError("Timed out waiting for ComfyUI /system_stats.")

print("ComfyUI is ready! Opening free Cloudflare quick tunnel...")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

printed_url = False
try:
    for line in tunnel_proc.stdout:
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match and not printed_url:
            printed_url = True
            url = match.group(0)
            print("\n" + "=" * 64)
            print("  CLEARFRAME GPU UPSCALER TUNNEL IS LIVE!")
            print("  Copy & paste this HTTPS URL into ClearFrame:")
            print(f"  ->  {url}")
            print("=" * 64 + "\n")
            print("Keep this cell running while ClearFrame processes your video.")
        if comfy_proc.poll() is not None:
            print("ComfyUI process stopped. Restarting Cell 2 is required.")
            break
finally:
    tunnel_proc.terminate()
    comfy_proc.terminate()